# Download, extract, & prep mcool files

In [ ]:
%%bash

cd ~/noble_lab/repos/Chromatin_Analysis_MOp/data/snm3C-seq/pseudobulk_3C/nobackup/Raw.100K

filenames=$(for each in Oligo_NN L6_CT_CTX_Glut Astro-TE_NN L2_3_IT_CTX_Glut Endo_NN L4_5_IT_CTX_Glut L5_IT_CTX_Glut L6_IT_CTX_Glut; do ls $each*; done)

mkdir originals
mv *.mcool originals/
mkdir selected && cd selected


for each in $filenames; do
	echo -e "\n======== $each"
	cooler cp ../originals/$each::/resolutions/100000 $each::/resolutions/100000 && cooler ls $each
done


for each in $(ls *.mcool); do
	echo -e "\n\n======== $each"
	cooler coarsen -k 25 $each::/resolutions/100000 -o $each::/resolutions/2500000 -a && echo && cooler ls $each
done


for each in $(ls *.mcool); do
	echo -e "\n======== $each"
	cooler balance $each::/resolutions/2500000 --cis-only --convergence-policy 'discard' 
done


# for each in $filenames; do
# 	echo -e "\n$each"
# 	cooler ls $each
# done


# Setup

In [ ]:
%load_ext autoreload
%autoreload 2 

In [ ]:
# import core packages
import warnings
warnings.filterwarnings("ignore")
from itertools import combinations
import os

# import semi-core packages
import matplotlib.pyplot as plt
from matplotlib import colors
%matplotlib inline
plt.style.use('seaborn-v0_8-poster')
import numpy as np
import pandas as pd
from multiprocessing import Pool

# import open2c libraries
import bioframe

import cooler
import cooltools

from packaging import version
if version.parse(cooltools.__version__) < version.parse('0.5.2'):
    raise AssertionError("tutorial relies on cooltools version 0.5.2 or higher,"+
                         "please check your cooltools version and update to the latest")

# count cpus
num_cpus = os.getenv('SLURM_CPUS_PER_TASK')
if not num_cpus:
    num_cpus = os.cpu_count()
num_cpus = int(num_cpus)

# Idk

In [ ]:
resolution = int(1000 * 1000 * 2.5) # 2.5 Mb (2.5e6 bp)

In [ ]:
# get centromere data from ucsc gap file
mm10_gap = pd.read_csv(
    '/net/gs/vol1/home/gesine/noble_lab/resources/genomic/gaps/ucsc.mm10.gap.txt',
    sep='\t', header=None, names=(
        'bin', 'chrom', 'start', 'end', 'ix', 'n', 'size', 'type', 'bridge'))
mm10_cens = mm10_gap.loc[mm10_gap.type == 'centromere', ['chrom', 'start', 'end']].copy().reset_index()
mm10_cens['mid'] = ((mm10_cens.start + mm10_cens.end) / 2).astype(int)
# Use bioframe to fetch the genomic features from the UCSC.
mm10_chromsizes = bioframe.fetch_chromsizes('mm10')
# create a view with chromosome arms using chromosome sizes and definition of centromeres
mm10_arms = bioframe.make_chromarms(mm10_chromsizes,  mm10_cens)
# remove chrX, chrY, chrM
mm10_arms = mm10_arms[~mm10_arms.chrom.isin(['chrX', 'chrY', 'chrM'])].reset_index(drop=True)
# remove arms that are less than the current binsize/resolution
mm10_arms = mm10_arms[(mm10_arms.end - mm10_arms.start) > resolution]

In [ ]:
# mcool_file = '/net/gs/vol1/home/gesine/noble_lab/repos/Chromatin_Analysis_MOp/data/snm3C-seq/pseudobulk_3C/nobackup/Raw.100K/L2_3_IT_CTX_Glut.Raw.100K.mcool'
mcool_file = '/net/gs/vol1/home/gesine/noble_lab/repos/Chromatin_Analysis_MOp/data/snm3C-seq/pseudobulk_3C/nobackup/Q.100K/selected/L2_3_IT_CTX_Glut.Q.100K.mcool'

uri = f'{mcool_file}::/resolutions/{resolution:g}'

clr = cooler.Cooler(uri)

cis_cov, tot_cov = cooltools.coverage(clr, ignore_diags=1, nproc=2)
print(f"nreads: {cis_cov.sum():,}", flush=True)

In [ ]:
cvd_smooth_agg = cooltools.expected_cis(
    clr=clr,
    view_df= mm10_arms[mm10_arms.chrom.isin(clr.chromnames)].reset_index(drop=True),
    smooth=True,
    aggregate_smoothed=True,
    smooth_sigma=0.1,
    nproc=num_cpus,
    ignore_diags=1, # GC: Given very coarse bin size, only ignore main diagonal
    intra_only=True,  # GC: snm3C-seq data only contains intra-chrom counts
    #clr_weight_name=None,
)

In [ ]:
import pandas as pd
import glob
import os
import ast
import numpy as np
from cooltools.sandbox.expected_smoothing import log_smooth


dir_matrix2d = '/net/gs/vol1/home/gesine/noble_lab/repos/Chromatin_Analysis_MOp/cluster.LINK/astro/compare_to_1cell/matrix2d'


sc_dis_intramol_file = glob.glob(os.path.join(dir_matrix2d, '*.distances.intramol.tsv.gz'))
if len(sc_dis_intramol_file) != 1:
    raise ValueError("Couldn't find unique file for intra-mol single cell distances")
sc_dis_intramol_file = sc_dis_intramol_file[0]

sc_dis_intramol = pd.read_csv(
    sc_dis_intramol_file, sep='\t', header=None, index_col=0,
    converters={0: ast.literal_eval})



sc_invdis_intramol = sc_dis_intramol.pow(-1)
sc_invdis_intramol.index = [j - i for i, j in sc_invdis_intramol.index]
sc_invdis_intramol.columns.name = 'genomic_dis'
sc_invdis_intramol.sort_index(inplace=True)

In [26]:
sc_invdis_intramol.shape
# res.T[(1, 'n')]

(44698, 3702)

In [32]:
def agg_across_genomic_dist(df):
    n_detected_sum = df.notnull().sum(axis=0)
    n_detected_sum.name = 'n'
    data_sum = df.sum(axis=0)
    data_sum.name = 'data'

    agg = pd.concat([
        data_sum.to_frame().T, n_detected_sum.to_frame().T])
    return agg
    

by_genomic_dis = sc_invdis_intramol.groupby(level=0).apply(agg_across_genomic_dist)

In [47]:
sigma_log10=0.1
window_sigma=5
points_per_sigma=10

col = 1

cell_data = by_genomic_dis[col].unstack()

smoothed_data, smoothed_n = log_smooth(
    cell_data.index.values.astype(np.float64),
    cell_data[['data', 'n']].values.T,
    sigma_log10=sigma_log10,
    window_sigma=window_sigma,
    points_per_sigma=points_per_sigma)

In [50]:
cell_data['smoothed_data'] = smoothed_data
cell_data['smoothed_n'] = smoothed_n

cell_data['smoothed_freq'] = smoothed_data / smoothed_n

cell_data

,n,data,smoothed_data,smoothed_n,smoothed_freq
1,23.0,50.129543,49.869668,23.010678,2.167240
2,24.0,25.753989,28.076187,24.154141,1.162376
3,25.0,37.610282,34.745875,24.239789,1.433423
4,23.0,33.750597,32.528586,23.228483,1.400375
5,23.0,30.387762,29.406181,22.614101,1.300347
...,...,...,...,...,...
72,0.0,0.000000,0.133522,0.435142,0.306848
73,0.0,0.000000,0.124410,0.409996,0.303443
74,0.0,0.000000,0.116112,0.386466,0.300446
75,0.0,0.000000,0.108468,0.364576,0.297519
